In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sqlalchemy import create_engine
from scipy.stats import linregress

plt.style.use('default')

In [ ]:
DB_USER = "postgres"
DB_PASSWORD = "password"
DB_HOST = "localhost"
DB_PORT = "5433"
DB_NAME = "DailyVitals"

connection_string = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(connection_string)

In [ ]:
query = """
SELECT
    e.exercise_session_id,
    p.first_name,
    et.exercise_name,
    e.start_time,
    e.duration_minutes,
    e.calories_expended,
    e.intensity
FROM exercise_session e
JOIN person p
    ON p.person_id = e.person_id
JOIN exercise_type et
    ON et.exercise_type_id = e.exercise_type_id
ORDER BY start_time;
"""

df = pd.read_sql(query, engine)

df.head()

In [ ]:
print("Sessions:", len(df))
print()

print("Total Calories Burned:",
      df["calories_expended"].sum())

print("Average Calories Per Session:",
      round(df["calories_expended"].mean(), 1))

print("Maximum Session:",
      df["calories_expended"].max())

print("Minimum Session:",
      df["calories_expended"].min())

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(
    df["start_time"],
    df["calories_expended"],
    linewidth=2
)

plt.title("Calories Burned Per Session")
plt.ylabel("Calories")
plt.xlabel("Date")

plt.grid(True)

plt.show()

In [ ]:
df["rolling_7"] = (
    df["calories_expended"]
    .rolling(window=7)
    .mean()
)

plt.figure(figsize=(12,6))

plt.plot(
    df["start_time"],
    df["calories_expended"],
    label="Actual"
)

plt.plot(
    df["start_time"],
    df["rolling_7"],
    linewidth=3,
    label="7 Session Average"
)

plt.legend()

plt.title("Exercise Trend")
plt.show()

In [ ]:
x = np.arange(len(df))

y = df["calories_expended"]

slope, intercept, r, p, std_err = linregress(x, y)

print("Trend Slope:", round(slope,2))

if slope > 10:
    print("Strong Upward Trend")
elif slope > 2:
    print("Moderate Upward Trend")
elif slope > -2:
    print("Flat Trend")
else:
    print("Declining Trend")

In [ ]:
df["month"] = pd.to_datetime(
    df["start_time"]
).dt.to_period("M")

monthly = (
    df.groupby("month")
      .agg(
          sessions=("exercise_session_id","count"),
          total_calories=("calories_expended","sum"),
          avg_calories=("calories_expended","mean")
      )
      .reset_index()
)

monthly

In [ ]:
plt.figure(figsize=(12,6))

plt.bar(
    monthly["month"].astype(str),
    monthly["total_calories"]
)

plt.title("Total Calories Burned Per Month")
plt.xticks(rotation=45)

plt.show()

In [ ]:
std = df["calories_expended"].std()

avg = df["calories_expended"].mean()

cv = std / avg

consistency = max(
    0,
    round(100 - (cv * 100))
)

print("Consistency Score:", consistency)

In [ ]:
best = df.nlargest(
    5,
    "calories_expended"
)

best[
    [
        "start_time",
        "exercise_name",
        "calories_expended"
    ]
]

In [ ]:
latest = df.iloc[-1]

summary = f"""
Exercise Summary

Sessions:
{len(df)}

Average Calories:
{df['calories_expended'].mean():.0f}

Latest Session:
{latest['calories_expended']:.0f}

Best Session:
{df['calories_expended'].max():.0f}

Total Calories Burned:
{df['calories_expended'].sum():,.0f}
"""

print(summary)